# MediaPipe Modern Computer Vision — Complete Guide
### Tasks API 2024/2025 · No Legacy `mp.solutions`

> **Projects covered:** Hand Tracking · Volume Control · Virtual Mouse · Gesture Control · Face Mesh · Pose Estimation

---

## Chapter 1 — Introduction to MediaPipe

MediaPipe is Google's open-source cross-platform framework for real-time computer vision and ML pipelines. It runs directly on your device — no cloud, no API key, no GPU required. Originally powering features in Google Meet and Pixel phones, it is now the industry standard for on-device hand tracking, face detection, and pose estimation.

### 1.1 Two Eras — The Most Important Thing to Know

MediaPipe has **two completely different APIs**. Every YouTube tutorial uses the OLD one. This guide teaches ONLY the new Tasks API.

| Old API (`mp.solutions`) — AVOID | New Tasks API — USE THIS |
|---|---|
| `import mediapipe as mp` | `from mediapipe.tasks import python` |
| `mp.solutions.hands.Hands()` | `HandLandmarker.create_from_options(opts)` |
| `results.multi_hand_landmarks` | `result.hand_landmarks` |
| Models bundled in package | Must download `.task` model files |
| Status: **DEPRECATED** | Status: **CURRENT** |

> ⚠️ **WARNING:** If a tutorial uses `mp.solutions` anywhere, it is outdated. The old API is deprecated and will be removed. Everything in this guide uses the Tasks API exclusively.

### 1.2 Projects You Will Build

1. **Finger Counter** — detect raised fingers in real time
2. **Volume Control** — pinch gesture controls system audio
3. **Virtual Mouse** — index finger moves cursor, pinch clicks
4. **Gesture Controller** — classify gestures to trigger actions
5. **Face Mesh** — visualize 468 facial landmarks
6. **Pose / Exercise Counter** — count reps using joint angles

---
## Chapter 2 — Environment Setup

### 2.1 Install Python & Packages

Requires **Python 3.8–3.11** (3.10 is ideal). Always use a virtual environment.

> 💡 **Apple Silicon (M1/M2/M3):** use `opencv-python-headless` instead of `opencv-python` to avoid ARM display driver conflicts.

In [ ]:
# Run this cell to install all required packages
# (In a terminal, create a venv first: python -m venv mediapipe_env)

# Core packages
# pip install --upgrade pip
# pip install mediapipe opencv-python numpy

# For Volume Control (Windows only):
# pip install pycaw comtypes

# For Virtual Mouse (all platforms):
# pip install pyautogui

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mediapipe', 'opencv-python', 'numpy', '-q'])

### 2.2 Download Model Files

The Tasks API **REQUIRES** separate `.task` model files. They are **NOT** included in the pip package. Download them once and reuse across all projects.

In [ ]:
import os, urllib.request

os.makedirs('models', exist_ok=True)

BASE = 'https://storage.googleapis.com/mediapipe-models'
MODELS = {
    'hand_landmarker.task':       f'{BASE}/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    'face_landmarker.task':       f'{BASE}/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
    'pose_landmarker_lite.task':  f'{BASE}/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task',
    'gesture_recognizer.task':    f'{BASE}/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task',
}

for fname, url in MODELS.items():
    dest = os.path.join('models', fname)
    if os.path.exists(dest):
        print(f'  [SKIP]  {fname} already exists')
    else:
        print(f'  [DL]    Downloading {fname}...')
        urllib.request.urlretrieve(url, dest)
        print(f'  [OK]    {fname}')

print('\nAll models ready!')

### 2.3 Project Folder Structure

```
mediapipe_projects/
├── models/
│   ├── hand_landmarker.task
│   ├── face_landmarker.task
│   ├── pose_landmarker_lite.task
│   └── gesture_recognizer.task
├── project1_finger_counter/   finger_counter.py
├── project2_volume_control/   volume_control.py
├── project3_virtual_mouse/    virtual_mouse.py
├── project4_gesture/          gesture_controller.py
├── project5_face_mesh/        face_mesh.py
└── project6_pose/             pose_counter.py
```

In [ ]:
# verify_install.py — run this first!
import sys, os

def chk(name, fn):
    try: fn(); print(f'  [OK]  {name}')
    except Exception as e: print(f'  [FAIL] {name}: {e}')

chk('mediapipe',       lambda: __import__('mediapipe'))
chk('cv2',             lambda: __import__('cv2'))
chk('numpy',           lambda: __import__('numpy'))
chk('mediapipe.tasks', lambda: __import__('mediapipe.tasks', fromlist=['tasks']))

for m in ['hand_landmarker', 'face_landmarker', 'gesture_recognizer']:
    chk(f'models/{m}.task', lambda f=m: open(f'models/{f}.task','rb').close())

import cv2
cap = cv2.VideoCapture(0)
chk('webcam', lambda: (cap.isOpened() or (_ for _ in ()).throw(Exception('no cam'))))
cap.release()

---
## Chapter 3 — OpenCV Fundamentals

OpenCV handles camera input and all drawing. MediaPipe only does detection math. Master this loop first.

### 3.1 The Core Camera Loop

In [ ]:
import cv2

cap = cv2.VideoCapture(0)  # 0 = default webcam, try 1/2 for external
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

while True:
    ret, frame = cap.read()    # ret = success bool, frame = numpy array
    if not ret: break

    frame = cv2.flip(frame, 1) # Mirror so it feels natural

    # ─── All processing goes HERE ───

    cv2.imshow('Window', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### 3.2 Coordinate System & Color

> **Key facts:**
> - OpenCV uses **BGR** (not RGB!) color order
> - MediaPipe Tasks API **requires RGB** — always convert
> - MediaPipe landmarks use **normalized coords** (0.0 to 1.0) — multiply by frame size to get pixels

In [ ]:
# Frame is a numpy array of shape (height, width, 3)
h, w = frame.shape[:2]   # e.g. 720, 1280

# OpenCV uses BGR (Blue, Green, Red) — NOT RGB!
# MediaPipe Tasks API REQUIRES RGB — always convert:
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# MediaPipe landmarks use NORMALIZED coords (0.0 to 1.0)
# Convert to pixel coords:
# px = int(landmark.x * w)
# py = int(landmark.y * h)

### 3.3 Drawing Functions

In [ ]:
# Circle: (image, center_xy, radius, color_BGR, thickness=-1 fills)
# cv2.circle(frame, (cx, cy), 10, (0, 255, 0), -1)     # filled green
# cv2.circle(frame, (cx, cy), 10, (0, 255, 0),  2)     # outline

# Line: (image, pt1, pt2, color, thickness)
# cv2.line(frame, (x1,y1), (x2,y2), (255, 165, 0), 3)

# Rectangle: (image, top_left, bottom_right, color, thickness)
# cv2.rectangle(frame, (10,10), (200,100), (0,0,255), -1)  # filled red

# Text: (image, text, origin, font, scale, color, thickness)
# cv2.putText(frame, 'Hello!', (50,60),
#     cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255), 2)

print('Drawing function signatures loaded (uncomment to use in a camera loop)')

### 3.4 FPS Counter

In [ ]:
import time
prev = 0

# Inside the while loop:
# now = time.time()
# fps = int(1 / (now - prev + 1e-9))
# prev = now
# cv2.putText(frame, f'FPS: {fps}', (w-130, 30),
#     cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

print('FPS counter snippet ready')

---
## Chapter 4 — Tasks API Architecture

Every MediaPipe task (hands, face, pose, gesture) follows the **exact same three-component pattern**. Learn this once, use it everywhere.

### 4.1 The Three-Component Pattern

1. **`BaseOptions`** — WHERE is the model? WHICH hardware?
2. **`TaskOptions`** — HOW should it behave?
3. **`create_from_options()`** — creates the task object

In [1]:
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import HandLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode  import VisionTaskRunningMode as VisionRunningMode

# 1. BaseOptions — WHERE is the model? WHICH hardware?
base_options = python.BaseOptions(
    model_asset_path='models/hand_landmarker.task'
)

# 2. TaskOptions — HOW should it behave?
# (on_result callback defined in section 4.4)
def on_result(result, output_image, timestamp_ms):
    pass  # placeholder — defined properly in 4.4

options = HandLandmarker.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    result_callback=on_result   # required for LIVE_STREAM
)

# 3. Create the task object
landmarker = HandLandmarker.create_from_options(options)
print('HandLandmarker created successfully!')
landmarker.close()

SyntaxError: invalid syntax (2382453476.py, line 3)

### 4.2 Three Running Modes

| Mode | Description | Call |
|---|---|---|
| `IMAGE` | Single static image, synchronous | `detector.detect(mp_image)` |
| `VIDEO` | Pre-recorded video with timestamps, synchronous | `detector.detect_for_video(img, ts_ms)` |
| `LIVE_STREAM` | Webcam real-time, asynchronous, callback required | `detector.detect_async(img, ts_ms)` |

> 📝 **NOTE:** For ALL webcam projects use `LIVE_STREAM`. It processes frames asynchronously for the best frame rate. Results come back ~1 frame later via the callback — imperceptible to the user.

### 4.3 Creating a MediaPipe Image

In [ ]:
import cv2, mediapipe as mp

# STEP 1: Convert BGR to RGB
# rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# STEP 2: Wrap in mp.Image
# mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

# STEP 3: Pass to detector
# landmarker.detect_async(mp_img, timestamp_ms)

print('mp.Image creation pattern loaded')

### 4.4 The LIVE_STREAM Callback Pattern

> ⚠️ **WARNING:** NEVER call `detect_async()` from inside the callback — it causes a **deadlock**. Only store the result in the callback and process it in the main loop.

In [ ]:
import time

# Shared storage for results (list makes it mutable in closure)
latest_result = [None]

# Callback — runs on a background thread
# ONLY store the result here — no heavy processing!
def on_result(result, output_image, timestamp_ms):
    latest_result[0] = result

# Build options with callback attached
options = HandLandmarker.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='models/hand_landmarker.task'),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=on_result,
    num_hands=2
)
landmarker = HandLandmarker.create_from_options(options)

# Main loop skeleton:
# start_time = time.time()
# while True:
#     ret, frame = cap.read()
#     frame = cv2.flip(frame, 1)
#
#     ts_ms = int((time.time() - start_time) * 1000)  # MUST increase every frame
#
#     rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
#     landmarker.detect_async(mp_img, ts_ms)
#
#     result = latest_result[0]
#     if result:
#         pass  # process result here

landmarker.close()
print('Callback pattern ready')

---
## Chapter 5 — Hand Landmarks In Depth

### 5.1 The 21 Landmarks

MediaPipe detects exactly **21 landmarks** per hand. Each has a fixed index.

| Index | Landmark |
|---|---|
| 0 | WRIST |
| 1–4 | THUMB: CMC, MCP, IP, **TIP** |
| 5–8 | INDEX: MCP, PIP, DIP, **TIP** |
| 9–12 | MIDDLE: MCP, PIP, DIP, **TIP** |
| 13–16 | RING: MCP, PIP, DIP, **TIP** |
| 17–20 | PINKY: MCP, PIP, DIP, **TIP** |

> 💡 **Memory trick:** Wrist=0. Each finger has 4 landmarks. 1 + 5×4 = 21 total.
> 
> **TIP indices:** Thumb=4, Index=8, Middle=12, Ring=16, Pinky=20  
> **PIP (middle knuckle):** Index=6, Middle=10, Ring=14, Pinky=18

In [ ]:
# 5.2 Accessing Landmarks

# result = latest_result[0]
#
# if result and result.hand_landmarks:
#     for hand_lms in result.hand_landmarks:
#         # hand_lms = list of 21 NormalizedLandmark objects
#         # Each has: .x (0-1), .y (0-1), .z (depth estimate)
#
#         thumb_tip = hand_lms[4]
#         index_tip = hand_lms[8]
#
#         # Convert to pixel coordinates:
#         h, w = frame.shape[:2]
#         tx, ty = int(thumb_tip.x * w), int(thumb_tip.y * h)
#         ix, iy = int(index_tip.x * w), int(index_tip.y * h)
#
#         cv2.circle(frame, (tx,ty), 10, (255,0,200), -1)
#         cv2.circle(frame, (ix,iy), 10, (0,255,200), -1)

# 5.3 Handedness
# if result and result.handedness:
#     for i, hlist in enumerate(result.handedness):
#         label = hlist[0].category_name  # 'Left' or 'Right'
#         score = hlist[0].score
#         # NOTE: Because we flip the frame, MediaPipe 'Left' = your visual RIGHT

print('Landmark access patterns loaded')

In [ ]:
# 5.4 Reusable Hand Utils
# Save as utils/hand_utils.py and import in your projects

import math, cv2

TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]

def lm_px(lm, frame):
    h, w = frame.shape[:2]
    return int(lm.x * w), int(lm.y * h)

def dist_px(lm1, lm2, frame):
    x1,y1 = lm_px(lm1, frame)
    x2,y2 = lm_px(lm2, frame)
    return math.hypot(x2-x1, y2-y1)

def count_fingers(lms):
    up = [lms[4].x < lms[3].x]  # thumb: horizontal check
    for tip, pip in zip(TIPS[1:], PIPS[1:]):
        up.append(lms[tip].y < lms[pip].y)  # others: vertical
    return sum(up), up

CONNECTIONS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
               (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
               (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)]

def draw_hand(frame, lms):
    h, w = frame.shape[:2]
    for a,b in CONNECTIONS:
        ax,ay = int(lms[a].x*w), int(lms[a].y*h)
        bx,by = int(lms[b].x*w), int(lms[b].y*h)
        cv2.line(frame,(ax,ay),(bx,by),(180,180,180),2)
    for lm in lms:
        cv2.circle(frame,(int(lm.x*w),int(lm.y*h)),6,(0,150,255),-1)

print('Hand utils defined: lm_px, dist_px, count_fingers, draw_hand')

---
## Chapter 6 — Project 1: Finger Counter

Detect raised fingers in real time.

**Logic:**
- A finger is **'up'** when its TIP has a smaller y-coordinate than its PIP knuckle (y grows downward)
- The **thumb** is special — compare x-coordinates instead (horizontal movement)

**Run from terminal:** `python project1_finger_counter/finger_counter.py`  
**Quit:** press `q`

In [ ]:
# project1_finger_counter/finger_counter.py
# ─────────────────────────────────────────
# Run this script directly in a terminal, not inside Jupyter
# (OpenCV imshow requires a display window)

import cv2, mediapipe as mp, time
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import HandLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

MODEL = 'models/hand_landmarker.task'
TIPS  = [4, 8, 12, 16, 20]
PIPS  = [3, 6, 10, 14, 18]

latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def count_fingers(lms):
    up = [lms[4].x < lms[3].x]  # thumb: horizontal check
    for tip,pip in zip(TIPS[1:], PIPS[1:]):
        up.append(lms[tip].y < lms[pip].y)
    return sum(up), up

def draw_hand(frame, lms):
    h,w = frame.shape[:2]
    edges=[(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
           (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
           (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)]
    for a,b in edges:
        cv2.line(frame,(int(lms[a].x*w),int(lms[a].y*h)),
                       (int(lms[b].x*w),int(lms[b].y*h)),(180,180,180),2)
    for lm in lms:
        cv2.circle(frame,(int(lm.x*w),int(lm.y*h)),6,(0,150,255),-1)

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = HandLandmarker.HandLandmarkerOptions(
        base_options=base, running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result, num_hands=2,
        min_hand_detection_confidence=0.5)
    det = HandLandmarker.create_from_options(opts)

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    t0 = time.time(); prev = 0

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]

        ts = int((time.time()-t0)*1000)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        det.detect_async(mp_img, ts)

        res = latest_result[0]
        total = 0
        if res and res.hand_landmarks:
            for lms in res.hand_landmarks:
                draw_hand(frame, lms)
                n, up = count_fingers(lms)
                total += n
                labels = ['Thumb','Index','Middle','Ring','Pinky']
                for i,(label,is_up) in enumerate(zip(labels, up)):
                    col = (0,200,0) if is_up else (0,0,200)
                    cv2.rectangle(frame,(10,60+i*44),(175,98+i*44),col,-1)
                    cv2.putText(frame,label,(20,87+i*44),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2)

        # Big count display
        cv2.rectangle(frame,(0,0),(105,58),(40,40,40),-1)
        cv2.putText(frame,str(total),(8,52),
            cv2.FONT_HERSHEY_SIMPLEX,2.1,(0,255,255),4)

        # FPS
        now=time.time()
        cv2.putText(frame,f'FPS:{int(1/(now-prev+1e-9))}',(w-130,30),
            cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)
        prev=now

        cv2.imshow('Finger Counter',frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); det.close()

if __name__=='__main__': main()

---
## Chapter 7 — Project 2: Volume Control

Measure the pixel distance between **thumb tip** (landmark 4) and **index tip** (landmark 8). Map that distance range to 0–100% volume using `numpy.interp()`.

- Small distance (pinch) = quiet
- Large distance (spread) = loud

> 💡 **Smoothing tip:** The `smooth = smooth*0.75 + target*0.25` line is an **exponential moving average**. It prevents volume from jumping when your hand trembles. Increase `0.75` for smoother (but slower) response.

In [ ]:
# 7.1 Distance-to-Volume Mapping
import math, numpy as np

MIN_DIST, MAX_DIST = 25, 220  # calibrate to your hand size

def get_dist(lm1, lm2, frame):
    h,w = frame.shape[:2]
    x1,y1 = int(lm1.x*w), int(lm1.y*h)
    x2,y2 = int(lm2.x*w), int(lm2.y*h)
    return math.hypot(x2-x1, y2-y1), (x1,y1), (x2,y2)

def dist_to_pct(dist):
    return int(np.clip(np.interp(dist,[MIN_DIST,MAX_DIST],[0,100]),0,100))

print('Distance utility functions defined')

In [ ]:
# 7.2 Volume API Snippets

# Windows (requires: pip install pycaw comtypes)
# from ctypes import cast, POINTER
# from comtypes import CLSCTX_ALL
# from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
# devices  = AudioUtilities.GetSpeakers()
# interface= devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
# vol      = cast(interface, POINTER(IAudioEndpointVolume))
# vr       = vol.GetVolumeRange()   # e.g. (-65.25, 0.0)
# MIN_DB, MAX_DB = vr[0], vr[1]
# def set_vol(pct):
#     db = MIN_DB + (MAX_DB-MIN_DB)*(pct/100.0)
#     vol.SetMasterVolumeLevel(db, None)

# macOS alternative:
# import subprocess
# subprocess.run(['osascript','-e',f'set volume output volume {pct}'])

# Linux alternative:
# subprocess.run(['amixer','sset','Master',f'{pct}%'])

print('Volume API snippets loaded (uncomment for your platform)')

In [ ]:
# project2_volume_control/volume_control.py  (Windows)
# ──────────────────────────────────────────
# Run from terminal. Requires: pip install pycaw comtypes

import cv2, mediapipe as mp, math, time, numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import HandLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode
# from ctypes import cast, POINTER
# from comtypes import CLSCTX_ALL
# from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

MODEL = 'models/hand_landmarker.task'
MIN_DIST, MAX_DIST = 25, 220

# Windows volume setup (uncomment on Windows):
# devices  = AudioUtilities.GetSpeakers()
# iface    = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
# vol_ctrl = cast(iface, POINTER(IAudioEndpointVolume))
# vr       = vol_ctrl.GetVolumeRange()
# MIN_DB, MAX_DB = vr[0], vr[1]

latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def draw_bar(frame, pct):
    x,y,bw,bh = 50,150,35,260
    cv2.rectangle(frame,(x,y),(x+bw,y+bh),(50,50,50),-1)
    fh = int(bh*pct/100)
    cv2.rectangle(frame,(x,y+bh-fh),(x+bw,y+bh),
                  (0,int(255*pct/100),int(255*(1-pct/100))),-1)
    cv2.rectangle(frame,(x,y),(x+bw,y+bh),(180,180,180),2)
    cv2.putText(frame,f'{pct}%',(x-5,y+bh+28),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = HandLandmarker.HandLandmarkerOptions(
        base_options=base, running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result, num_hands=1)
    det  = HandLandmarker.create_from_options(opts)
    cap  = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
    t0 = time.time(); smooth = 50.0

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame,1)
        h, w  = frame.shape[:2]
        ts    = int((time.time()-t0)*1000)
        mp_img= mp.Image(image_format=mp.ImageFormat.SRGB,
                         data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        det.detect_async(mp_img, ts)

        res = latest_result[0]
        if res and res.hand_landmarks:
            lms   = res.hand_landmarks[0]
            th,ix_= lms[4], lms[8]
            tx,ty = int(th.x*w), int(th.y*h)
            ix,iy = int(ix_.x*w), int(ix_.y*h)
            dist  = math.hypot(ix-tx, iy-ty)
            target= int(np.clip(np.interp(dist,[MIN_DIST,MAX_DIST],[0,100]),0,100))
            smooth= smooth*0.75 + target*0.25  # exponential moving average

            # Set system volume (Windows):
            # db = MIN_DB + (MAX_DB-MIN_DB)*(smooth/100.0)
            # vol_ctrl.SetMasterVolumeLevel(db, None)

            cv2.line(frame,(tx,ty),(ix,iy),(255,165,0),3)
            cv2.circle(frame,(tx,ty),14,(255,0,200),-1)
            cv2.circle(frame,(ix,iy),14,(255,0,200),-1)
            cv2.circle(frame,((tx+ix)//2,(ty+iy)//2),10,(0,255,100),-1)

        draw_bar(frame, int(smooth))
        cv2.imshow('Volume Control', frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); det.close()

if __name__=='__main__': main()

---
## Chapter 8 — Project 3: Virtual Mouse

- **Index finger** (landmark 8) moves the cursor
- **Pinching** thumb + index triggers a click
- `numpy.interp()` maps camera coords → screen coords
- A reduced **'active zone'** (`REDUCE` margin) makes it easier to reach screen edges without moving your hand to extremes

**Constants to tune:**
- `REDUCE` — active zone margin in pixels (larger = smaller active region, easier to hit edges)
- `CLICK_T` — pinch distance threshold for click trigger
- `SMOOTH` — cursor smoothing factor (higher = smoother but laggier)

In [ ]:
# project3_virtual_mouse/virtual_mouse.py
# ────────────────────────────────────────
# Run from terminal. Requires: pip install pyautogui

import cv2, mediapipe as mp, math, time, numpy as np, pyautogui
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import HandLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

pyautogui.FAILSAFE = False; pyautogui.PAUSE = 0
MODEL   = 'models/hand_landmarker.task'
CAM_W, CAM_H = 1280, 720
SW, SH  = pyautogui.size()
REDUCE  = 100    # active zone margin in px
CLICK_T = 40     # px threshold to trigger click
SMOOTH  = 6      # cursor smoothing factor

latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = HandLandmarker.HandLandmarkerOptions(
        base_options=base, running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result, num_hands=1,
        min_hand_detection_confidence=0.65)
    det = HandLandmarker.create_from_options(opts)
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
    t0 = time.time(); mx,my = 0,0; clicking = False

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame,1)
        ts = int((time.time()-t0)*1000)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        det.detect_async(mp_img, ts)

        # Show active zone rectangle
        cv2.rectangle(frame,(REDUCE,REDUCE),(CAM_W-REDUCE,CAM_H-REDUCE),(0,100,255),2)

        res = latest_result[0]
        if res and res.hand_landmarks:
            lms  = res.hand_landmarks[0]
            idx  = lms[8]; th = lms[4]
            ix,iy= int(idx.x*CAM_W), int(idx.y*CAM_H)
            tx,ty= int(th.x*CAM_W),  int(th.y*CAM_H)

            # Map camera coords to screen coords
            sx = int(np.clip(np.interp(ix,[REDUCE,CAM_W-REDUCE],[0,SW]),0,SW-1))
            sy = int(np.clip(np.interp(iy,[REDUCE,CAM_H-REDUCE],[0,SH]),0,SH-1))
            mx += (sx-mx)//SMOOTH; my += (sy-my)//SMOOTH
            pyautogui.moveTo(mx, my)

            # Click on pinch
            dist = math.hypot(ix-tx, iy-ty)
            if dist < CLICK_T:
                if not clicking: pyautogui.click(); clicking = True
                cv2.circle(frame,(ix,iy),20,(0,255,0),-1)
                cv2.putText(frame,'CLICK',(ix-25,iy-30),
                    cv2.FONT_HERSHEY_SIMPLEX,0.75,(0,255,0),2)
            else:
                clicking = False
                cv2.circle(frame,(ix,iy),12,(255,165,0),-1)
            cv2.circle(frame,(tx,ty),10,(200,0,200),-1)

        cv2.imshow('Virtual Mouse', frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); det.close()

if __name__=='__main__': main()

---
## Chapter 9 — Project 4: Gesture Recognizer

The `GestureRecognizer` model classifies **7 standard gestures** out of the box. Uses `recognize_async()` instead of `detect_async()`.

### Built-in Gestures

| Gesture Name | Description |
|---|---|
| `None` | No recognized gesture |
| `Closed_Fist` | All fingers curled |
| `Open_Palm` | Flat open hand |
| `Pointing_Up` | Index finger up |
| `Thumb_Down` | Thumbs down |
| `Thumb_Up` | Thumbs up |
| `Victory` | Peace sign |
| `ILoveYou` | Pinky + index + thumb |

In [ ]:
# project4_gesture/gesture_controller.py
# ───────────────────────────────────────
# Run from terminal.

import cv2, mediapipe as mp, time
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import GestureRecognizer
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

MODEL    = 'models/gesture_recognizer.task'
COOLDOWN = 1.5  # seconds between triggers

ACTIONS = {
    'Thumb_Up':    ('VOLUME UP',   (0,200,0)),
    'Thumb_Down':  ('VOLUME DOWN', (0,0,200)),
    'Open_Palm':   ('PLAY/PAUSE',  (200,200,0)),
    'Victory':     ('NEXT TRACK',  (200,0,200)),
    'Closed_Fist': ('PREV TRACK',  (0,200,200)),
}

latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = GestureRecognizer.GestureRecognizerOptions(
        base_options=base, running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result, num_hands=1)
    rec  = GestureRecognizer.create_from_options(opts)
    cap  = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
    t0 = time.time(); last_t = 0
    msg, msg_col = 'Show a gesture', (200,200,200)

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame,1)
        h,w = frame.shape[:2]
        ts  = int((time.time()-t0)*1000)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        rec.recognize_async(mp_img, ts)  # NOTE: recognize_async, not detect_async!

        res = latest_result[0]; now = time.time()
        if res and res.gestures:
            name  = res.gestures[0][0].category_name
            score = res.gestures[0][0].score
            cv2.putText(frame,f'Gesture: {name} ({score:.0%})',(10,55),
                cv2.FONT_HERSHEY_SIMPLEX,1.2,(0,255,255),3)
            if name in ACTIONS and score>0.80 and now-last_t>COOLDOWN:
                action, msg_col = ACTIONS[name]
                msg   = f'{name}: {action}'
                last_t= now
                print(f'[ACTION] {action}')
                # Add actual media control API calls here

        # Fade action banner at bottom
        if now-last_t < 2.5:
            ov = frame.copy()
            cv2.rectangle(ov,(0,h-90),(w,h),(0,0,0),-1)
            frame = cv2.addWeighted(ov,0.6,frame,0.4,0)
            cv2.putText(frame,msg,(w//2-200,h-35),
                cv2.FONT_HERSHEY_SIMPLEX,1.3,msg_col,3)

        cv2.imshow('Gesture Controller', frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); rec.close()

if __name__=='__main__': main()

---
## Chapter 10 — Face Landmarker: 468 Points

`FaceLandmarker` detects **468 landmarks** across the face plus optional **52 blendshapes** (expression coefficients).

### Key Landmark Indices

| Index | Location |
|---|---|
| 1 | Nose tip |
| 9 | Chin bottom |
| 33 | Right eye outer corner |
| 133 | Right eye inner corner |
| 362 | Left eye inner corner |
| 263 | Left eye outer corner |
| 61 | Right lip corner |
| 291 | Left lip corner |
| 0 | Upper lip center |
| 17 | Lower lip center |

In [ ]:
# project5_face_mesh/face_mesh.py
# ────────────────────────────────
# Run from terminal.

import cv2, mediapipe as mp, time
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import FaceLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

MODEL = 'models/face_landmarker.task'
latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = FaceLandmarker.FaceLandmarkerOptions(
        base_options=base,
        running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,   # 52 expression values
    )
    det = FaceLandmarker.create_from_options(opts)
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    t0 = time.time()

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame,1)
        h, w = frame.shape[:2]
        ts = int((time.time()-t0)*1000)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        det.detect_async(mp_img, ts)

        result = latest_result[0]
        if result and result.face_landmarks:
            for face_lms in result.face_landmarks:
                # Draw all 468 dots
                for lm in face_lms:
                    cv2.circle(frame,(int(lm.x*w),int(lm.y*h)),1,(0,255,0),-1)

        if result and result.face_blendshapes:
            bs = {b.category_name: b.score for b in result.face_blendshapes[0]}

            # Expression detection
            expressions = []
            if bs.get('jawOpen',0) > 0.6:       expressions.append('MOUTH OPEN')
            if bs.get('mouthSmileLeft',0)>0.4 and bs.get('mouthSmileRight',0)>0.4:
                                                 expressions.append('SMILING')
            if bs.get('eyeBlinkLeft',0)>0.5 and bs.get('eyeBlinkRight',0)>0.5:
                                                 expressions.append('EYES CLOSED')

            for i, expr in enumerate(expressions):
                cv2.putText(frame, expr, (10, 50+i*35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2)

        cv2.imshow('Face Mesh', frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); det.close()

if __name__=='__main__': main()

### Useful Blendshapes Reference

| Blendshape | What it detects |
|---|---|
| `jawOpen` | Mouth open |
| `mouthSmileLeft` / `mouthSmileRight` | Smiling |
| `eyeBlinkLeft` / `eyeBlinkRight` | Wink / blink |
| `eyebrowInnerUp` | Surprise |
| `browDownLeft` / `browDownRight` | Frown / concentration |
| `cheekPuff` | Puffed cheeks |

---
## Chapter 11 — Pose Landmarker: Full Body

`PoseLandmarker` detects **33 body landmarks** from nose to feet. Three model tiers:
- **lite** — fastest
- **full** — balanced  
- **heavy** — most accurate

### Key Landmark Indices

| Index | Landmark |
|---|---|
| 0 | NOSE |
| 11 | LEFT_SHOULDER |
| 12 | RIGHT_SHOULDER |
| 13 | LEFT_ELBOW |
| 14 | RIGHT_ELBOW |
| 15 | LEFT_WRIST |
| 16 | RIGHT_WRIST |
| 23 | LEFT_HIP |
| 24 | RIGHT_HIP |
| 25 | LEFT_KNEE |
| 26 | RIGHT_KNEE |
| 27 | LEFT_ANKLE |
| 28 | RIGHT_ANKLE |

In [ ]:
# 11.2 Joint Angle Utility
import numpy as np

def joint_angle(a, b, c, frame):
    '''Angle at joint B formed by A-B-C. Returns degrees 0-180.'''
    h,w = frame.shape[:2]
    ba = np.array([a.x*w-b.x*w, a.y*h-b.y*h])
    bc = np.array([c.x*w-b.x*w, c.y*h-b.y*h])
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-9)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

print('joint_angle(a, b, c, frame) -> degrees')

In [ ]:
# project6_pose/pose_counter.py  — Bicep Curl Rep Counter
# ──────────────────────────────
# Run from terminal.

import cv2, mediapipe as mp, time, numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import PoseLandmarker
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

MODEL = 'models/pose_landmarker_lite.task'
latest_result = [None]
def on_result(r,o,t): latest_result[0] = r

def joint_angle(a, b, c, frame):
    h,w = frame.shape[:2]
    ba = np.array([a.x*w-b.x*w, a.y*h-b.y*h])
    bc = np.array([c.x*w-b.x*w, c.y*h-b.y*h])
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-9)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

def main():
    base = python.BaseOptions(model_asset_path=MODEL)
    opts = PoseLandmarker.PoseLandmarkerOptions(
        base_options=base, running_mode=VisionRunningMode.LIVE_STREAM,
        result_callback=on_result, num_poses=1,
        min_pose_detection_confidence=0.5)
    det = PoseLandmarker.create_from_options(opts)
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    t0 = time.time()

    reps = 0; stage = None  # rep counter state

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame,1)
        ts = int((time.time()-t0)*1000)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        det.detect_async(mp_img, ts)

        result = latest_result[0]
        if result and result.pose_landmarks:
            pose  = result.pose_landmarks[0]
            # Left arm: shoulder(11) -> elbow(13) -> wrist(15)
            angle = joint_angle(pose[11], pose[13], pose[15], frame)

            if angle > 160:   curr = 'DOWN'
            elif angle < 35:  curr = 'UP'
            else:             curr = stage

            if curr == 'DOWN' and stage == 'UP': reps += 1
            stage = curr

            cv2.putText(frame,f'Reps: {reps}',(10,60),
                cv2.FONT_HERSHEY_SIMPLEX,2,(0,255,0),4)
            cv2.putText(frame,f'{stage} | {int(angle)}deg',(10,120),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),3)

        cv2.imshow('Pose Counter', frame)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cap.release(); cv2.destroyAllWindows(); det.close()

if __name__=='__main__': main()

---
## Chapter 12 — Performance & Troubleshooting

### 12.1 Optimization Tips

| Technique | Effect |
|---|---|
| Lower resolution (640×480) | Fastest single change |
| Lite model for pose | `pose_landmarker_lite` vs full/heavy |
| `num_hands=1` | Halves compute vs `num_hands=2` |
| GPU acceleration | `delegate=Delegate.GPU` |
| Disable blendshapes | `output_face_blendshapes=False` if unused |
| Frame skipping | Process every 2nd frame for heavy tasks |

In [ ]:
# GPU acceleration
# base_options = python.BaseOptions(
#     model_asset_path=MODEL,
#     delegate=python.BaseOptions.Delegate.GPU  # falls back to CPU gracefully
# )

# Frame skip — process every other frame
# frame_n = 0
# while True:
#     ret, frame = cap.read(); frame_n += 1
#     if frame_n % 2 == 0:
#         det.detect_async(mp_img, ts)
#     cv2.imshow('Win', frame)  # always show latest result

print('Performance snippets loaded')

### 12.2 Common Errors & Fixes

In [ ]:
import os

# ── Error 1: Model file not found ──────────────────────────────────────
print('Current directory:', os.getcwd())
print('hand_landmarker.task exists:', os.path.exists('models/hand_landmarker.task'))

# Use absolute path to avoid CWD confusion:
# MODEL = os.path.join(os.path.dirname(__file__), 'models', 'hand_landmarker.task')

# ── Error 2: Timestamps not monotonically increasing ───────────────────
# WRONG:  ts = frame_count          (not wall-clock time)
# CORRECT: ts_ms = int((time.time() - start) * 1000)

# ── Error 3: Deadlock from callback ────────────────────────────────────
# WRONG:  def on_result(r,o,t): landmarker.detect_async(...)  # DEADLOCK!
# CORRECT: def on_result(r,o,t): latest_result[0] = r         # only store

# ── Error 4: No landmarks detected ─────────────────────────────────────
# 1. Ensure hand is clearly in frame with good lighting
# 2. Confirm frame flip: frame = cv2.flip(frame, 1)
# 3. Confirm BGR→RGB conversion before mp.Image()
# 4. Lower confidence: min_hand_detection_confidence=0.3

# ── Error 5: Camera won't open ──────────────────────────────────────────
import cv2
for i in range(5):
    cap = cv2.VideoCapture(i)
    if cap.isOpened(): print(f'Camera found at index {i}'); cap.release(); break
else:
    print('No camera found at indices 0-4')
# Windows DirectShow backend: cv2.VideoCapture(0, cv2.CAP_DSHOW)

---
## Chapter 13 — API Quick Reference

### 13.2 Method Names Per Task

| Task | IMAGE | VIDEO | LIVE_STREAM |
|---|---|---|---|
| HandLandmarker | `detect` | `detect_for_video` | `detect_async` |
| FaceLandmarker | `detect` | `detect_for_video` | `detect_async` |
| PoseLandmarker | `detect` | `detect_for_video` | `detect_async` |
| GestureRecognizer | `recognize` | `recognize_for_video` | `recognize_async` |

In [ ]:
# 13.1 All Task Options — Reference

# HandLandmarker
# HandLandmarker.HandLandmarkerOptions(
#     base_options, running_mode, num_hands=2,
#     min_hand_detection_confidence=0.5,
#     min_hand_presence_confidence=0.5,
#     min_tracking_confidence=0.5,
#     result_callback=on_result)

# FaceLandmarker
# FaceLandmarker.FaceLandmarkerOptions(
#     base_options, running_mode, num_faces=1,
#     min_face_detection_confidence=0.5,
#     output_face_blendshapes=False,
#     output_facial_transformation_matrixes=False,
#     result_callback=on_result)

# PoseLandmarker
# PoseLandmarker.PoseLandmarkerOptions(
#     base_options, running_mode, num_poses=1,
#     min_pose_detection_confidence=0.5,
#     output_segmentation_masks=False,
#     result_callback=on_result)

# GestureRecognizer
# GestureRecognizer.GestureRecognizerOptions(
#     base_options, running_mode, num_hands=1,
#     min_hand_detection_confidence=0.5,
#     result_callback=on_result)

print('All task option signatures loaded')

In [ ]:
# 13.3 Import Template — copy-paste starter for any new project

import cv2, mediapipe as mp, time
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import HandLandmarker  # swap for Face/Pose/Gesture
from mediapipe.tasks.python.vision.core.vision_task_running_mode \
    import VisionTaskRunningMode as VisionRunningMode

MODEL = 'models/hand_landmarker.task'
latest_result = [None]

def on_result(r, o, t): latest_result[0] = r

base = python.BaseOptions(model_asset_path=MODEL)
opts = HandLandmarker.HandLandmarkerOptions(
    base_options=base,
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=on_result,
    num_hands=1
)
det = HandLandmarker.create_from_options(opts)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
t0  = time.time()

# while True:
#     ret, frame = cap.read()
#     if not ret: break
#     frame = cv2.flip(frame, 1)
#     ts    = int((time.time()-t0)*1000)
#     mp_img= mp.Image(image_format=mp.ImageFormat.SRGB,
#                      data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
#     det.detect_async(mp_img, ts)
#
#     result = latest_result[0]
#     if result and result.hand_landmarks:
#         pass  # your logic here
#
#     cv2.imshow('App', frame)
#     if cv2.waitKey(1)&0xFF==ord('q'): break
#
# cap.release(); cv2.destroyAllWindows(); det.close()

det.close()
print('Import template ready — uncomment the while loop to run')

---
## Chapter 14 — Learning Roadmap

### Week 1 — Foundations
- **Day 1:** Setup (Ch 2). Run `verify_install.py` — all green.
- **Day 2:** OpenCV loop (Ch 3). Draw shapes, display FPS.
- **Day 3:** Tasks API concepts (Ch 4). Understand callback pattern.
- **Day 4–5:** Hand landmarks (Ch 5). Print all 21 coords to terminal.
- **Day 6–7:** Project 1 — Finger Counter. Debug until stable.

### Week 2 — Core Projects
- **Day 8–9:** Volume Control. Calibrate `MIN/MAX_DIST` for your hand size.
- **Day 10–11:** Virtual Mouse. Fine-tune `REDUCE` and `SMOOTH` constants.
- **Day 12–13:** Gesture Controller. Test all 7 built-in gestures.
- **Day 14:** Code review. Add error handling and comments to all projects.

### Week 3 — Advanced Tasks
- **Day 15–16:** Face Mesh. Visualize 468 dots; experiment with blendshapes.
- **Day 17–18:** Pose Counter. Build bicep curl + squat counters.
- **Day 19–20:** Performance. Hit 30+ FPS on every project.
- **Day 21:** Multi-task pipeline — run hands + face simultaneously.

### Week 4 — Build Something Original
- Brainstorm a project combining 2+ tasks
- Build iteratively — get basic version working first, then add features
- Add `README.md` and push to GitHub

### Project Ideas to Explore Next
- Custom gesture training with MediaPipe Model Maker
- Sign language fingerspelling detector
- Attention tracker using gaze and head pose
- Real-time yoga pose coach with angle feedback
- Hands-free presentation slide controller
- AI drawing canvas — finger as paintbrush
- Blink-to-click accessible interface

---

> 💡 **The single most important habit:** Run every code example yourself and change **one thing at a time**. When something breaks, you learn more from the debugging than from the working code.
>
> Good luck, and enjoy building!